In [1]:
import pandas as pd
import numpy as np

In [2]:
from cleaning_pipeline import clean_agent_data

In [3]:
print(clean_agent_data)

<function clean_agent_data at 0x7a223989cb80>


# A4 — Agent Data Cleanup

## Data Quality and Cleaning Pipeline

This notebook cleans and validates agent transaction data using a reusable
cleaning pipeline. It checks data quality issues, merges transaction and
agent information, validates the results, and produces summary tables.

## 1. Data Quality Audit

The audit identifies data-quality issues across the six dimensions:

1. Completeness
2. Validity
3. Accuracy
4. Consistency
5. Uniqueness
6. Timeliness

In [4]:
audit = pd.DataFrame(
    columns=[
        "issue",
        "dimension",
        "column",
        "rows affected"
    ]
)

audit

,issue,dimension,column,rows affected


In [5]:
txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

print("Transactions:", txns.shape)
print("Agents:", agents.shape)

Transactions: (1001, 5)
Agents: (50, 3)


In [6]:
%run generate_data.py

Generated agent records: 50
Generated transaction records: 1001


/content/generate_data.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1,500' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  txns.loc[25, "amount"] = "1,500"


In [7]:
txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

print("Transaction rows:", len(txns))
print("Agent rows:", len(agents))

Transaction rows: 1001
Agent rows: 50


In [8]:
txns.head()

,txn_id,agent_id,txn_type,amount,txn_date
0,T00001,A020,Deposit,10430.022363203314,2026-01-01
1,T00002,A015,Withdrawal,28292.2981797781,2026-01-02
2,T00003,A020,Deposit,26589.5975649178,2026-01-03
3,T00004,A046,Withdrawal,16187.292394404276,2026-01-04
4,T00005,A039,Withdrawal,4629.461094470892,2026-01-05


In [9]:
clean_df, log = clean_agent_data(txns, agents)

print("Rows before cleaning:", len(txns))
print("Rows after cleaning:", len(clean_df))

print("\nCleaning log:")
for item in log:
    print("-", item)

Rows before cleaning: 1001
Rows after cleaning: 999

Cleaning log:
- Removed 1 duplicate transaction rows
- Found 1 invalid transaction types
- Imputed 2 missing or invalid amounts with the median
- Flagged 1 amount outliers for review
- Found 0 invalid transaction dates
- Found 1 transactions with unknown agents


In [10]:
clean_df.head()

,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,agent_name,county
0,T00001,A020,Deposit,10430.022363,2026-01-01,False,False,Agent 20,Nakuru
1,T00002,A015,Withdrawal,28292.29818,2026-01-02,False,False,Agent 15,Nairobi
2,T00003,A020,Deposit,26589.597565,2026-01-03,False,False,Agent 20,Nakuru
3,T00004,A046,Withdrawal,16187.292394,2026-01-04,False,False,Agent 46,Nairobi
4,T00005,A039,Withdrawal,4629.461094,2026-01-05,False,False,Agent 39,Uasin Gishu


In [11]:
clean_df.columns

Index(['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date',
       'amount_imputed', 'is_outlier', 'agent_name', 'county'],
      dtype='object')

In [12]:
audit_rows = [
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows affected": txns["txn_id"].duplicated().sum()
    },
    {
        "issue": "Missing or invalid amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows affected": pd.to_numeric(
            txns["amount"].astype("string").str.replace(",", "", regex=False),
            errors="coerce"
        ).isna().sum()
    },
    {
        "issue": "Invalid transaction types",
        "dimension": "Validity",
        "column": "txn_type",
        "rows affected": (
            ~txns["txn_type"].astype("string").str.strip().str.title()
            .isin(["Deposit", "Withdrawal"])
        ).sum()
    },
    {
        "issue": "Unknown agents",
        "dimension": "Accuracy",
        "column": "agent_id",
        "rows affected": (~txns["agent_id"].isin(agents["agent_id"])).sum()
    }
]

audit = pd.DataFrame(audit_rows)

audit

,issue,dimension,column,rows affected
0,Duplicate transaction IDs,Uniqueness,txn_id,1
1,Missing or invalid amounts,Completeness,amount,2
2,Invalid transaction types,Validity,txn_type,1
3,Unknown agents,Accuracy,agent_id,1


In [13]:
print("Transaction rows before cleaning:", len(txns))
print("Cleaned rows:", len(clean_df))

unknown_agents = clean_df["county"].isna().sum()

print("Transactions with unknown agents:", unknown_agents)

Transaction rows before cleaning: 1001
Cleaned rows: 999
Transactions with unknown agents: 1


In [14]:
county_summary = (
    clean_df
    .groupby("county")
    .agg(
        transactions=("txn_id", "count"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean")
    )
    .reset_index()
)

county_summary

,county,transactions,total_amount,average_amount
0,Kisumu,239,5858235.587562,24511.445973
1,Mombasa,89,2448576.489286,27512.095385
2,Nairobi,214,5713947.154876,26700.68764
3,Nakuru,189,4850315.818226,25663.046657
4,Uasin Gishu,267,6851785.961707,25662.119707


In [15]:
clean_df["week"] = clean_df["txn_date"].dt.to_period("W").astype(str)

weekly_pivot = pd.pivot_table(
    clean_df,
    index="week",
    columns="txn_type",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

weekly_pivot

txn_type,Deposit,Withdrawal
week,,
2025-12-29/2026-01-04,37019.619928,44479.590574
2026-01-05/2026-01-11,172843.31435,8041.543293
2026-01-12/2026-01-18,137637.401235,90265.211187
2026-01-19/2026-01-25,57958.93296,103475.238668
2026-01-26/2026-02-01,116711.854112,98929.284143
...,...,...
2028-08-28/2028-09-03,48435.856554,112325.82602
2028-09-04/2028-09-10,169846.698742,10674.108184
2028-09-11/2028-09-17,56023.51656,159392.603794


# Data Quality Report

## 1. Completeness
Missing or invalid transaction amounts were identified and handled through median imputation. The `amount_imputed` flag records which transactions were imputed.

## 2. Validity
Transaction types were standardized and restricted to the valid categories `Deposit` and `Withdrawal`. Amount values were converted to numeric values.

## 3. Accuracy
Transaction records were merged with the agent dataset using `agent_id`. Transactions without a matching agent were identified as unknown agents.

## 4. Consistency
Transaction types were standardized to a consistent format. Transaction dates were converted to datetime values for consistent analysis.

## 5. Uniqueness
Duplicate transactions were identified using `txn_id` and removed so that each transaction ID is unique.

## 6. Timeliness
Transaction dates were converted into a consistent datetime format, allowing transactions to be analyzed by week.

## Cleaning Outcome
The cleaned dataset contains the required quality-control fields, including `amount_imputed` and `is_outlier`. Outliers were flagged for review rather than automatically deleted.

In [16]:
import os

os.makedirs("outputs", exist_ok=True)

clean_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("Clean CSV created successfully!")

Clean CSV created successfully!


In [17]:
import os

print(os.path.exists("outputs/agent_transactions_clean.csv"))

True


In [18]:
clean_check = pd.read_csv("outputs/agent_transactions_clean.csv")

print("Rows:", len(clean_check))
print("Columns:", list(clean_check.columns))

Rows: 999
Columns: ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'agent_name', 'county', 'week']


In [19]:
required_columns = [
    "txn_id",
    "agent_id",
    "txn_type",
    "amount",
    "txn_date",
    "amount_imputed",
    "is_outlier",
    "county"
]

for column in required_columns:
    print(column, "✓" if column in clean_check.columns else "✗")

txn_id ✓
agent_id ✓
txn_type ✓
amount ✓
txn_date ✓
amount_imputed ✓
is_outlier ✓
county ✓


In [20]:
print("Duplicate txn_id:", clean_check["txn_id"].duplicated().sum())
print("Transaction types:", clean_check["txn_type"].unique())
print("Amount data type:", clean_check["amount"].dtype)
print("Unknown counties:", clean_check["county"].isna().sum())

Duplicate txn_id: 0
Transaction types: ['Deposit' 'Withdrawal']
Amount data type: float64
Unknown counties: 1


In [21]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 2.2.3
NumPy version: 2.1.3
